# 03 — BPE from scratch on the toy corpus (Part C)

The algorithm in five small pieces. In this notebook you write each piece as a 🧩 exercise **with the toy corpus**; you then package them in `src/bpe.py`, make `pytest tests/test_bpe.py -x` green, and run the experiments.

Corpus: `low low low lower lower / newer newer newest newest`

**How the 🧩 exercises work.** Cells marked 🧩 contain `____` blanks: complete them using the hints (they name the Python functions you need). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right. When an exercise matches a function of `src/`, the docstring of that function says so: you then **move your working code into `src/`** and run `pytest`.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent          # notebooks/ is one level below the repo root
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

from collections import Counter
from src.bpe import BPETokenizer, END_OF_WORD

In [ ]:
toy = (ROOT / "data/toy/toy_corpus.txt").read_text(encoding="utf-8").splitlines()
print(toy)

## The algorithm, step by step

```
1. word table:   {('l','o','w','</w>'): 3, ...}
2. count adjacent pairs, weighted by word frequency
3. pick the most frequent pair (ties: smallest pair)
4. merge it everywhere
5. repeat; keep the ordered list of merges
6. to encode a NEW word: start from its characters and apply the merges in the order they were learned
```

### 🧩 Exercise 3.1 — Step 1 — the word table

Turn the corpus into a table `{tuple of symbols: frequency}`. Each word is a tuple of its characters followed by the end-of-word marker.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `text.split()` gives the words; `Counter(words)` counts them.
* `tuple("low")` is `('l', 'o', 'w')`; add the marker with `+ ("</w>",)` (a one-element tuple needs the trailing comma).
* Dict comprehension: `{key: value for word, freq in counts.items()}`.

In [ ]:
words = " ".join(toy).split()
counts = ____(words)
table = {____: f for w, f in counts.items()}
for symbols, freq in table.items():
    print(freq, "x", " ".join(symbols))

In [ ]:
assert table[("l", "o", "w", "</w>")] == 3 and len(table) == 4
assert table[("n", "e", "w", "e", "s", "t", "</w>")] == 2
print("✅ Exercise 3.1 OK")

### 🧩 Exercise 3.2 — Step 2 — count the adjacent pairs (weighted)

Count every pair of neighbouring symbols, **adding the frequency of the word** each time (a pair occurring twice in a word counts twice).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `zip(symbols, symbols[1:])` yields the neighbouring pairs `(s0, s1), (s1, s2), …`.
* A `Counter` supports `pairs[key] += amount` even for missing keys.
* `pairs.most_common(5)` shows the five most frequent.

In [ ]:
def get_pairs(table):
    pairs = Counter()
    for symbols, freq in table.items():
        for a, b in ____:
            pairs[(a, b)] ____
    return pairs

pairs = get_pairs(table)
print(pairs.most_common(6))

In [ ]:
assert pairs[("w", "e")] == 6 and pairs[("l", "o")] == 5 and pairs[("w", "</w>")] == 3
assert sum(pairs.values()) == sum(freq * (len(s) - 1) for s, freq in table.items())
print("✅ Exercise 3.2 OK")

### 🧩 Exercise 3.3 — Step 3 — choose the best pair (with a tie rule)

Pick the pair with the **highest frequency**; if several pairs share it, pick the **lexicographically smallest** pair. (Without a fixed rule, two correct implementations disagree.)

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `min(iterable, key=function)` returns the element with the smallest key.
* To get 'highest frequency first, then smallest pair', use the tuple key `(-frequency, pair)`.
* Tuples compare element by element: `('e','we') < ('n','e')`.

In [ ]:
best = ____(pairs, key=lambda p: (____, ____))
print(best, pairs[best])

tie_example = Counter({("b", "c"): 4, ("a", "z"): 4, ("a", "b"): 3})
print(min(tie_example, key=lambda p: (-tie_example[p], p)))

In [ ]:
assert best == ("w", "e")
assert min(tie_example, key=lambda p: (-tie_example[p], p)) == ("a", "z")
print("✅ Exercise 3.3 OK")

### 🧩 Exercise 3.4 — Step 4 — merge a pair everywhere

Return a **new** table where every occurrence of `pair` is replaced by the concatenated symbol. Scan each word left to right (so `a a a` merged on `(a, a)` gives `aa a`).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Walk through the symbols with an index `i`: if `symbols[i], symbols[i+1]` equal the pair, append the merged symbol and advance by **2**, otherwise append `symbols[i]` and advance by **1**.
* The merged symbol is `pair[0] + pair[1]`.
* Words that become identical must have their frequencies added: `new[key] = new.get(key, 0) + freq`.
* Never modify the input table.

In [ ]:
def merge_pair(pair, table):
    new = {}
    for symbols, freq in table.items():
        out, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == ____ and symbols[i + 1] == ____:
                out.append(____)
                i += ____
            else:
                out.append(symbols[i])
                i += ____
        key = tuple(out)
        new[key] = new.get(key, 0) + freq
    return new

after = merge_pair(best, table)
for symbols, freq in after.items():
    print(freq, "x", " ".join(symbols))

In [ ]:
assert after[("l", "o", "we", "r", "</w>")] == 2
assert merge_pair(("a", "a"), {("a", "a", "a", "</w>"): 1}) == {("aa", "a", "</w>"): 1}
assert sum(after.values()) == sum(table.values())
assert table[("l", "o", "w", "</w>")] == 3          # input untouched
print("✅ Exercise 3.4 OK")

### 🧩 Exercise 3.5 — Step 5 — the training loop

Repeat *count → choose → merge* for 5 steps (or until no pair is left), recording each chosen pair in `merges`.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Stop early when `get_pairs(table)` is empty: `if not pairs: break`.
* Use the tie rule from exercise 3.3.
* Replace `table` by the merged table at each step (exercise 3.4).

In [ ]:
table = {tuple(w) + ("</w>",): f for w, f in Counter(words).items()}
merges = []
for step in range(5):
    pairs = get_pairs(table)
    if not ____:
        break
    best = min(pairs, key=lambda p: (-pairs[p], p))
    merges.append(____)
    table = ____
    print(f"step {step + 1}: merge {best} (frequency {pairs[best]})")

In [ ]:
assert merges == [("w", "e"), ("l", "o"), ("e", "we"), ("n", "ewe"), ("r", "</w>")]
print("✅ Exercise 3.5 OK")

**Hand check (required):** compute the first two merges on paper and compare. Explain why step 3 is a tie and how the rule resolved it.

**Your answer:**

_(write here)_

### 🧩 Exercise 3.6 — Step 6 — apply the merges to a NEW word

Encode a word that may not be in the training corpus: start from its characters + `</w>`, then repeatedly merge the adjacent pair that was **learned earliest** (lowest rank).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `merge_ranks = {pair: rank}` — build it with `enumerate(merges)`.
* Among the adjacent pairs of the current word that appear in `merge_ranks`, take the one with the **smallest rank**: `min(candidates)` works on `(rank, position)` tuples.
* To merge that pair in the word, reuse `merge_pair` on a one-word table: `merge_pair(pair, {tuple(symbols): 1})`, then take its single key.
* Stop when no adjacent pair is in `merge_ranks`.

In [ ]:
merge_ranks = {pair: rank for rank, pair in ____}

def encode_word(word):
    symbols = list(word) + ["</w>"]
    while len(symbols) > 1:
        candidates = [(merge_ranks[p], i) for i, p in enumerate(zip(symbols, symbols[1:])) if p in merge_ranks]
        if not candidates:
            break
        rank, _ = ____
        pair = ____
        symbols = list(next(iter(merge_pair(pair, {tuple(symbols): 1}))))
    return symbols

for w in ["low", "newer", "lowest", "zebra"]:
    print(w, "->", encode_word(w))

In [ ]:
assert encode_word("low") == ["lo", "w", "</w>"]
assert encode_word("newer") == ["newe", "r</w>"]
assert encode_word("zebra")[:2] == ["z", "e"]               # unseen characters stay separate
print("✅ Exercise 3.6 OK")

## ⏸ Stop: package your code in `src/bpe.py`

| Exercise | → `BPETokenizer` method |
|---|---|
| 3.1 | `initialize_vocabulary` |
| 3.2 | `get_pair_frequencies` |
| 3.3 + 3.5 | `train` (also record `merge_frequencies`, `merge_ranks`, the vocabulary) |
| 3.4 | `merge_pair` |
| 3.6 | `encode_word`; then write `encode` and `decode` |

Work through `pytest tests/test_bpe.py -x` from top to bottom. When it is green, run the experiments below with **your** class.

## C.4 Experiments — BPE-10, BPE-30, BPE-50

In [ ]:
unseen = ["lowest", "slower", "newly", "wider", "lowers", "weer"]
results, tokenizers = [], {}
for n in [10, 30, 50]:
    t = BPETokenizer().train(toy, num_merges=n)
    tokenizers[n] = t
    toks = [t.encode(w) for w in " ".join(toy).split()]
    results.append({"experiment": f"BPE-{n}", "merges requested": n, "merges learned": len(t.merges),
                    "vocab size (incl. [UNK])": t.vocab_size,
                    "avg tokens / training word": sum(map(len, toks)) / len(toks),
                    "avg tokens / unseen word": sum(len(t.encode(w)) for w in unseen) / len(unseen)})
pd.DataFrame(results)

In [ ]:
for n, t in tokenizers.items():
    print(f"=== BPE-{n}: learned merge rules (in order)"); print(" ", [f"{a}+{b}" for a, b in t.merges])
    print("  training corpus:", t.encode(toy[0]), "...")
    for w in unseen: print(f"  unseen {w!r:10} ->", t.encode(w))

**Observe & explain**
- Why are BPE-30 and BPE-50 identical?
- Look at `lowest` and `weer`: how does BPE represent unseen words? What happens with an unseen *character* (`t.encode("z")`)?
- Convert tokens to ids with `t.tokens_to_ids(...)` and back (next cell).

**Your answer:**

_(write here)_

In [ ]:
t = tokenizers[30]
toks = t.encode("lowest newer"); ids = t.tokens_to_ids(toks)
print(toks); print(ids); print(t.ids_to_tokens(ids)); print(repr(t.decode(toks)))

## C.5 Theory questions
1. Why does BPE merge frequent adjacent pairs? 2. Why must word frequencies be considered? 3. What happens when the number of merges increases?
4. Does increasing the number of merges always improve tokenization? 5. Token vs. merge rule?
6. Why does the order of merge rules matter? (concrete example: `test_merge_order_matters`)
7. How can BPE represent words not in its training corpus? 8. Character-level BPE vs. byte-level BPE?

**Your answer:**

_(write here)_